# Lab Task 1: Spaceship Titanic - Kaggle Competition

1. Load the data
2. Explore the data (EDA)
3. Clean and preprocess the data
4. Train a machine learning model
5. Make predictions and create submission file

## Required Libraries Import


In [ ]:

import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')

print("All libraries imported successfully!")

ModuleNotFoundError: No module named 'matplotlib'

## Loading Data


In [ ]:

train_data = pd.read_csv('training.csv')


test_data = pd.read_csv('test.csv')


sample_submission = pd.read_csv('sample_submission.csv')

print("Data loaded successfully!")
print(f"Training data has {train_data.shape[0]} rows and {train_data.shape[1]} columns")
print(f"Testing data has {test_data.shape[0]} rows and {test_data.shape[1]} columns")

In [ ]:
# Look at the first 5 rows of training data
print("First 5 rows of training data:")
train_data.head()

In [ ]:
# Get information about the data
print("Information about the training data:")
train_data.info()

In [ ]:
# Get statistical summary of numerical columns
print("Statistical summary:")
train_data.describe()

In [ ]:
# Check for missing values in training data
print("Missing values in training data:")
missing_values = train_data.isnull().sum()
print(missing_values)

## Step 4: Data Visualization
creating some charts to understand data better

In [ ]:

plt.figure(figsize=(8, 5))
sns.countplot(x='Transported', data=train_data)
plt.title('Distribution of Transported (Target Variable)')
plt.xlabel('Transported')
plt.ylabel('Count')
plt.show()

In [ ]:
# Visualize missing values using a heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(train_data.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

## Step 5: Data Preprocessing
Now let's clean our data and prepare it for the model

In [ ]:
# Drop columns that are not useful for prediction

columns_to_drop = ['PassengerId', 'Cabin', 'Name']

# Make copies of our data
train_clean = train_data.drop(columns=columns_to_drop)
test_clean = test_data.drop(columns=columns_to_drop)

print("Columns after dropping:")
print(train_clean.columns.tolist())

In [ ]:
# Fill missing values
# For numerical columns, fill with median (middle value)
# For categorical columns, fill with mode (most common value)

# Numerical columns
numerical_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for col in numerical_cols:
    median_value = train_clean[col].median()
    train_clean[col] = train_clean[col].fillna(median_value)
    test_clean[col] = test_clean[col].fillna(median_value)

# Categorical columns
categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

for col in categorical_cols:
    mode_value = train_clean[col].mode()[0]
    train_clean[col] = train_clean[col].fillna(mode_value)
    test_clean[col] = test_clean[col].fillna(mode_value)

print("Missing values after filling:")
print(train_clean.isnull().sum())

In [ ]:
label_encoder = LabelEncoder()

# Columns to encode
cols_to_encode = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

for col in cols_to_encode:
    # Combine train and test to ensure consistent encoding
    combined = pd.concat([train_clean[col], test_clean[col]])
    label_encoder.fit(combined)
    
    train_clean[col] = label_encoder.transform(train_clean[col])
    test_clean[col] = label_encoder.transform(test_clean[col])

print("Data after encoding:")
train_clean.head()

## Step 6: Train Machine Learning Model
Using Random Forest Classifier it's like asking many decision trees and taking a vote

In [ ]:
X = train_clean.drop('Transported', axis=1)
y = train_clean['Transported']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# We keep 20% for testing our model before final submission

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

In [ ]:
# Create and train the Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

print("Model training complete!")

## Step 7: Evaluate Model Performance


In [ ]:
# Make predictions on validation data
val_predictions = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, val_predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

## Step 8: Make Predictions and submission File

In [ ]:
# Make predictions on test data
test_predictions = model.predict(test_clean)

# Create submission dataframe
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': test_predictions
})

# Show first few rows
submission.head()

In [ ]:
# Save to CSV file
submission.to_csv('my_submission.csv', index=False)
print("Submission file saved! Upload 'my_submission.csv' to Kaggle")